In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer,AutoConfig,AutoModel
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

In [2]:
 model_ckpt = 'bert-base-uncased'
 model = AutoModel.from_pretrained(model_ckpt)

 print(model)
 print(model.embeddings)
 print(model.encoder)
 tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
 config = AutoConfig.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
# input
print(config.vocab_size)
#768 = 64 * 12
print(config.hidden_size)
token_embedding = nn.Embedding(config.vocab_size,config.hidden_size)
token_embedding

30522
768


Embedding(30522, 768)

In [12]:
sample_text = 'time flies like an arrow'
model_inputs = tokenizer(sample_text,return_tensors='pt',add_special_tokens=False)
model_inputs

{'input_ids': tensor([[ 2051, 10029,  2066,  2019,  8612]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [13]:
input_embeddings = token_embedding(model_inputs['input_ids'])
input_embeddings.shape

torch.Size([1, 5, 768])

# 下三角矩阵

In [15]:
import math
q = k =v = input_embeddings
#从而 由 1，5，768 * 1，768，5 得到 1，5，5
scores = torch.bmm(q,k.transpose(1,2))/math.sqrt(k.size(-1))
scores

tensor([[[ 2.6857e+01,  1.8917e-01,  4.5892e-01, -3.8125e-01,  1.3474e+00],
         [ 1.8917e-01,  2.7285e+01, -2.2344e-02, -6.4059e-01, -1.0338e+00],
         [ 4.5892e-01, -2.2344e-02,  2.6369e+01, -3.1278e-03, -5.0213e-01],
         [-3.8125e-01, -6.4059e-01, -3.1278e-03,  3.0965e+01, -3.0855e-01],
         [ 1.3474e+00, -1.0338e+00, -5.0213e-01, -3.0855e-01,  2.7763e+01]]],
       grad_fn=<DivBackward0>)

In [17]:
model_inputs['input_ids'].shape

torch.Size([1, 5])

In [16]:
seq_len = model_inputs['input_ids'].size(-1)

#上三角,如果是下三角，则为triu
mask = torch.tril(torch.ones(seq_len,seq_len)).unsqueeze(0)
print(mask.shape)
mask

torch.Size([1, 5, 5])


tensor([[[1., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [1., 1., 1., 0., 0.],
         [1., 1., 1., 1., 0.],
         [1., 1., 1., 1., 1.]]])

In [20]:
torch.exp(torch.tensor(-float('inf')))

tensor(0.)

In [18]:
scores.masked_fill(mask==0,-float('inf'))

tensor([[[ 2.6857e+01,        -inf,        -inf,        -inf,        -inf],
         [ 1.8917e-01,  2.7285e+01,        -inf,        -inf,        -inf],
         [ 4.5892e-01, -2.2344e-02,  2.6369e+01,        -inf,        -inf],
         [-3.8125e-01, -6.4059e-01, -3.1278e-03,  3.0965e+01,        -inf],
         [ 1.3474e+00, -1.0338e+00, -5.0213e-01, -3.0855e-01,  2.7763e+01]]],
       grad_fn=<MaskedFillBackward0>)

In [19]:
scores.masked_fill_(mask==0,-float('inf'))

tensor([[[ 2.6857e+01,        -inf,        -inf,        -inf,        -inf],
         [ 1.8917e-01,  2.7285e+01,        -inf,        -inf,        -inf],
         [ 4.5892e-01, -2.2344e-02,  2.6369e+01,        -inf,        -inf],
         [-3.8125e-01, -6.4059e-01, -3.1278e-03,  3.0965e+01,        -inf],
         [ 1.3474e+00, -1.0338e+00, -5.0213e-01, -3.0855e-01,  2.7763e+01]]],
       grad_fn=<MaskedFillBackward0>)

In [21]:
def scaled_dot_product_attn(q,k,v,mask=None):
  # q,k,v:(b,s,h/a) (a: number heads,12,768/12=64)
  dim_k = k.size(-1)
  #b s s
  scores = torch.bmm(q,k.transpose(1,2))/math.sqrt(dim_k)
  if mask is not None:
    scores.masked_fill_(mask ==0 ,-float('inf'))
  #令上对角为0
  attn_weights = F.softmax(scores,dim=-1)
  print(attn_weights)
  return torch.bmm(attn_weights,v)

In [22]:
scaled_dot_product_attn(q,k,v,mask)

tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [1.7071e-12, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [5.5896e-12, 3.4543e-12, 1.0000e+00, 0.0000e+00, 0.0000e+00],
         [2.4356e-14, 1.8792e-14, 3.5548e-14, 1.0000e+00, 0.0000e+00],
         [3.3733e-12, 3.1183e-13, 5.3066e-13, 6.4401e-13, 1.0000e+00]]],
       grad_fn=<SoftmaxBackward0>)


tensor([[[ 3.4475,  1.7280,  0.5106,  ..., -0.9379,  0.2764,  0.0352],
         [-1.1267,  0.0244,  0.8563,  ...,  1.0878, -0.4107,  1.1531],
         [ 0.5607,  0.1539, -0.4317,  ..., -1.1436,  0.8922,  0.8319],
         [-0.0384,  3.2923, -0.3340,  ...,  0.9623,  1.3704, -0.3725],
         [-1.1959,  0.3004,  0.8208,  ...,  0.5305, -0.2567, -0.5145]]],
       grad_fn=<BmmBackward0>)